# Test chỉ số thời gian time_decay

In [12]:
import numpy as np
import pandas as pd
import time

In [33]:
from surprise import SVD, CoClustering, NMF, SlopeOne, KNNBaseline, BaselineOnly, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate

In [34]:
path = "movie_data/"

In [35]:
df_items = pd.read_csv(path + "movies.csv")
df_interactions = pd.read_csv(path + "ratings.csv")

In [36]:
df_items.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [37]:
df_interactions.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [38]:
df_interactions[df_interactions['userId'] == 1]

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
227,1,3744,4.0,964980694
228,1,3793,5.0,964981855
229,1,3809,4.0,964981220
230,1,4006,4.0,964982903


In [39]:
def add_time_features(df):
    # Tạo bản sao để tránh báo lỗi SettingWithCopyWarning
    df = df.copy()
    
    # 1. Tính thời điểm xem cuối cùng (max timestamp) của mỗi người dùng
    # transform('max') sẽ trả về giá trị max lặp lại cho mọi dòng của userId đó
    df['last_watched_at'] = df.groupby('userId')['timestamp'].transform('max')
    
    # 2. Tính số ngày chênh lệch (day_diff)
    # 1 ngày = 24 giờ * 3600 giây = 86,400 giây
    df['day_diff'] = (df['last_watched_at'] - df['timestamp']) / 86400
    
    # Làm tròn để dễ nhìn (tùy chọn)
    df['day_diff'] = df['day_diff'].round(2)
    
    return df

In [40]:
df_interactions = add_time_features(df_interactions)
df_interactions.head()

,userId,movieId,rating,timestamp,last_watched_at,day_diff
0,1,1,4.0,964982703,965719662,8.53
1,1,3,4.0,964981247,965719662,8.55
2,1,6,4.0,964982224,965719662,8.54
3,1,47,5.0,964983815,965719662,8.52
4,1,50,5.0,964982931,965719662,8.53


In [41]:
# Xác định decay factor lambda dựa trên nửa đời (half-life)
HALF_LIFE_DAYS = 30
decay_lambda = np.log(2) / HALF_LIFE_DAYS

print(f"Decay factor (lambda) for a half-life of {HALF_LIFE_DAYS} days: {decay_lambda:.6f}")

Decay factor (lambda) for a half-life of 30 days: 0.023105


In [42]:
def cal_score_on_time(df, lam):
    df = df.copy()

    df['heuristic_score'] = df['rating'] * np.exp(-lam * df['day_diff'])

    return df

In [43]:
df_interactions = cal_score_on_time(df_interactions, decay_lambda)
df_interactions.head()

,userId,movieId,rating,timestamp,last_watched_at,day_diff,heuristic_score
0,1,1,4.0,964982703,965719662,8.53,3.284484
1,1,3,4.0,964981247,965719662,8.55,3.282966
2,1,6,4.0,964982224,965719662,8.54,3.283725
3,1,47,5.0,964983815,965719662,8.52,4.106554
4,1,50,5.0,964982931,965719662,8.53,4.105605


In [44]:
# Algorithm 1: Phân loại thể loại yêu thích và không thích của người dùng

def get_user_genres_preferences(user_id, df_interaction, df_items):
  # Get movieId which user rated
  user_ratings = df_interactions[df_interaction['userId'] == user_id]

  # Count genre
  genre_counts = Counter()
  for m_id in user_ratings['movieId']:
    # Get string genres of movie ("Action|Adventure")
    genre_str = df_items.loc[df_items['movieId'] == m_id, 'genres'].values[0]
    if pd.isna(genre_str): continue
    for genre in genre_str.split('|'):
      genre_counts[genre] += 1

  # Get all genre
  all_genres = set()
  df_items['genres'].str.split('|').apply(lambda x: all_genres.update(x) if isinstance(x, list) else None)

  # Sort asc
  sorted_genres = sorted(list(all_genres), key=lambda x: genre_counts.get(x, 0), reverse=True)

  # Find mid-point
  mid_point = len(sorted_genres) // 2
  fav_genres = sorted_genres[:mid_point]
  unfav_genres = sorted_genres[mid_point:]

  return fav_genres, unfav_genres


In [45]:
# Algorithm 2: Kiểm tra tính liên quan (Relevance) dựa trên Genre
def is_movie_relevant(movie_id, df_items, fav_genres, unfav_genres):
    res = df_items.loc[df_items['movieId'] == movie_id, 'genres'].values
    if len(res) == 0 or pd.isna(res[0]): return False

    movie_genres = [g.strip() for g in res[0].split('|')]
    # Score = (Số genre thích) - (Số genre ghét)
    score = sum(1 for g in movie_genres if g in fav_genres) - \
            sum(1 for g in movie_genres if g in unfav_genres)
    return score > 0

In [46]:
# Algorithm 4: Tính các chỉ số Precision@K, Recall@K, F1 score
def calculate_metrics_at_k(predictions, df_interaction, df_items, k):
  user_est_true = {}
  for uid, iid, true_r, est, _ in predictions:
    if uid not in user_est_true:
      user_est_true[uid] = []
    user_est_true[uid].append((iid, est))

  precisions = []
  recalls = []

  for uid, user_ratings in user_est_true.items():
    # Lấy gu của user
    fav, unfav = get_user_genres_preferences(uid, df_interaction, df_items)

    # Sắp xếp dự đoán theo điểm 'est' cao nhất
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_k = user_ratings[:k]

    # Tính TP (True Positives)
    n_rel_and_rec_k = sum(1 for (iid, est) in top_k if is_movie_relevant(iid, df_items, fav, unfav))

    # Tính tổng số phim relevant thực tế có trong tập test của user
    n_rel = sum(1 for (iid, est) in user_ratings if is_movie_relevant(iid, df_items, fav, unfav))

    # Tính precision and recall cho user
    precisions.append(n_rel_and_rec_k / k)
    recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

  # Lấy avg
  avg_prec = np.mean(precisions)
  avg_rec = np.mean(recalls)

  f1 = (2 * avg_prec * avg_rec) / (avg_rec + avg_prec) if (avg_rec + avg_prec) >0 else 0

  return avg_prec, avg_rec, f1

In [47]:
# Danh sách các thuật toán
algorithms = [
    SVD(),
    CoClustering(),
    NMF(),
    SlopeOne(),
    KNNBaseline(),
    BaselineOnly()
]

In [48]:
# Chia dữ liệu Trainning và test, tỉ lệ 75/25
reader = Reader(rating_scale=(0.5, 5))

data = Dataset.load_from_df(df_interactions[['userId', 'movieId', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.25)

In [52]:
from surprise import accuracy
from collections import Counter

In [53]:
# Huấn luyện, dự đoán và đo thời gian
results_table = []

for algorithm in algorithms:
  algo_name = algorithm.__class__.__name__

  start_time = time.time()

  # Train
  algorithm.fit(trainset)

  # Predict
  predictions = algorithm.test(testset)

  #Time
  eval_time = time.time() - start_time


  # Tinh RMSE va MAE
  rmse_val = accuracy.rmse(predictions, verbose=False)
  mae_val = accuracy.mae(predictions, verbose=False)

  # Tinh Precision, recall, f1
  prec, rec, f1 = calculate_metrics_at_k(predictions, df_interactions, df_items, k = 15)

  results_table.append([algo_name, rmse_val, mae_val, prec, rec, f1, eval_time])
  print(f"{algo_name:<15} | {rmse_val:<8.4f} | {mae_val:<8.4f} | {prec:<10.4f} | {rec:<8.4f} | {f1:<8.4f} | {eval_time:<8.4f}")


SVD             | 0.8699   | 0.6695   | 0.7423     | 0.6803   | 0.7099   | 1.0900  
CoClustering    | 0.9375   | 0.7272   | 0.7436     | 0.6805   | 0.7107   | 2.1679  
NMF             | 0.9215   | 0.7073   | 0.7398     | 0.6796   | 0.7084   | 2.3260  
SlopeOne        | 0.8941   | 0.6838   | 0.7402     | 0.6791   | 0.7083   | 6.4595  
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
KNNBaseline     | 0.8683   | 0.6646   | 0.7395     | 0.6795   | 0.7082   | 2.2207  
Estimating biases using als...
BaselineOnly    | 0.8656   | 0.6674   | 0.7384     | 0.6791   | 0.7075   | 0.2654  


In [54]:

# Xuất ra DataFrame để hiển thị đẹp
df_final_results = pd.DataFrame(results_table, columns=['Algorithms', 'RMSE', 'MAE', 'Precision', 'Recall', 'F1 Score', 'Time'])
df_final_results

,Algorithms,RMSE,MAE,Precision,Recall,F1 Score,Time
0,SVD,0.869910,0.669453,0.742295,0.680262,0.709926,1.089990
1,CoClustering,0.937532,0.727233,0.743607,0.680495,0.710652,2.167861
2,NMF,0.921548,0.707307,0.739781,0.679567,0.708397,2.325958
3,SlopeOne,0.894105,0.683802,0.740219,0.679091,0.708338,6.459472
4,KNNBaseline,0.868279,0.664613,0.739454,0.679487,0.708203,2.220677
5,BaselineOnly,0.865608,0.667390,0.738361,0.679127,0.707506,0.265397
